# Image Feature Extraction **[DEMO]**

Determine which device to run PyTorch on:
- CUDA if an Nvidia GPU is installed
- CPU otherwise

In [1]:
import torch

# Set device to CUDA if we have an Nvidia GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {device}")

Using device cpu


---
## Example 1

Load image datasets

In [2]:
from PIL import Image
import requests

img_urls = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.png",
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/cats.jpeg",
]
image_real = Image.open(requests.get(img_urls[0], stream=True).raw).convert("RGB")
image_gen = Image.open(requests.get(img_urls[1], stream=True).raw).convert("RGB")

Import pretrained image processor and ML model

In [3]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = AutoModel.from_pretrained("google/vit-base-patch16-224").to(device)

c:\Users\Installer\.virtualenvs\Computer-Vision-Pipeline-tJFGsvW2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Installer\.virtualenvs\Computer-Vision-Pipeline-tJFGsvW2\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Installer\.cache\huggingface\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Develope

Basic inference function

In [6]:
def infer(image):
    inputs = processor(image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    return outputs.pooler_output

Pass images to inference function to obtain embeddings

In [ ]:
embed_real = infer(image_real)
embed_gen = infer(image_gen)

tensor([[-7.8319e-01, -6.4497e-02,  3.6043e-01, -3.8960e-01,  4.9951e-01,
         -3.0066e-01,  4.4539e-01, -4.2413e-01,  4.9492e-01, -3.1164e-01,
         -3.2057e-01, -1.0222e-01,  2.6053e-01,  3.4325e-01, -2.9206e-01,
          6.6675e-01,  1.8092e-01, -2.4628e-01, -4.0941e-01,  2.1447e-01,
         -3.1086e-01, -7.8936e-01,  7.8109e-01,  2.7253e-01,  3.2406e-01,
          1.1563e-01, -3.7704e-01, -4.0147e-01,  2.4517e-01, -7.6820e-01,
         -2.2212e-01, -1.1326e-01, -5.3125e-01, -7.4888e-01, -3.2273e-01,
         -5.1137e-01,  6.5916e-03, -3.2908e-02, -3.4901e-01, -3.5147e-01,
          1.7570e-01, -1.9028e-02, -2.2268e-01, -1.9111e-01, -4.6987e-01,
          2.8725e-01,  6.2831e-01, -4.2359e-01,  1.7984e-02,  2.8010e-01,
         -5.5973e-01,  7.6020e-01,  5.4247e-01,  2.3555e-01, -3.4808e-01,
         -6.7052e-01,  2.9118e-01,  2.6204e-01, -6.2514e-01,  3.7569e-01,
          1.1134e-01,  1.9076e-01,  2.5212e-01, -4.6977e-01, -6.1460e-01,
          7.3872e-01,  6.8673e-01, -1.

Calculate similarity scores

In [8]:
from torch.nn.functional import cosine_similarity

similarity_score = cosine_similarity(embed_real, embed_gen, dim=1)
print(similarity_score)

tensor([0.5799], grad_fn=<SumBackward1>)


---
## Example 2